IMPORTS

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


Load Dataset

In [17]:
df = pd.read_csv("../data/dataset.csv")
df.head()


,num_loops,max_loop_depth,has_recursion,uses_list,uses_dict,uses_set,lines_of_code,num_functions,uses_append,uses_pop,pattern_label,efficiency_label,source
0,0,0,0,0,0,0,5,1,0,0,recursion,inefficient,generic
1,2,2,0,0,0,0,7,1,0,0,bruteforce,inefficient,generic
2,3,3,0,0,0,0,10,1,0,0,bruteforce,inefficient,generic
3,2,2,0,0,0,0,12,1,0,0,bruteforce,inefficient,generic
4,3,3,0,0,0,0,10,1,0,0,bruteforce,inefficient,generic


Loop_density

In [18]:
# --------------------------------------------------
# FEATURE ENGINEERING: LOOP DENSITY
# --------------------------------------------------

# Avoid division by zero just in case
df["loop_density"] = df["num_loops"] / df["lines_of_code"].replace(0, 1)

# Quick sanity check
df[["num_loops", "lines_of_code", "loop_density"]].head()


,num_loops,lines_of_code,loop_density
0,0,5,0.000000
1,2,7,0.285714
2,3,10,0.300000
3,2,12,0.166667
4,3,10,0.300000


In [19]:
# --------------------------------------------------
# GROUP PATTERN LABELS INTO MAJOR CATEGORIES
# --------------------------------------------------

def group_pattern(label):
    if label in ["hashindex", "hashing", "dict", "counter"]:
        return "hashing"
    elif label in ["bruteforce", "nestedscan"]:
        return "brute_force"
    elif label in ["recursive", "helper"]:
        return "recursion"
    elif label in ["stack", "simulation"]:
        return "stack"
    elif label in ["twopointer"]:
        return "two_pointer"
    elif label in ["dp", "prefix"]:
        return "dynamic_programming"
    else:
        return "other"

df["pattern_grouped"] = df["pattern_label"].apply(group_pattern)

# Check distribution
df["pattern_grouped"].value_counts()


pattern_grouped
other                  19
brute_force            11
recursion              10
hashing                10
stack                   9
dynamic_programming     8
two_pointer             5
Name: count, dtype: int64

DataSet Sanity Check

In [20]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   num_loops         72 non-null     int64  
 1   max_loop_depth    72 non-null     int64  
 2   has_recursion     72 non-null     int64  
 3   uses_list         72 non-null     int64  
 4   uses_dict         72 non-null     int64  
 5   uses_set          72 non-null     int64  
 6   lines_of_code     72 non-null     int64  
 7   num_functions     72 non-null     int64  
 8   uses_append       72 non-null     int64  
 9   uses_pop          72 non-null     int64  
 10  pattern_label     72 non-null     object 
 11  efficiency_label  72 non-null     object 
 12  source            72 non-null     object 
 13  loop_density      72 non-null     float64
 14  pattern_grouped   72 non-null     object 
dtypes: float64(1), int64(10), object(4)
memory usage: 8.6+ KB


,num_loops,max_loop_depth,has_recursion,uses_list,uses_dict,uses_set,lines_of_code,num_functions,uses_append,uses_pop,loop_density
count,72.000000,72.000000,72.0,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.00000,72.000000
mean,1.277778,1.152778,0.0,0.402778,0.111111,0.013889,11.361111,0.930556,0.180556,0.12500,0.139392
std,0.981838,0.816377,0.0,0.493899,0.316475,0.117851,7.123272,0.678158,0.387349,0.33304,0.136155
min,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
25%,1.000000,1.000000,0.0,0.000000,0.000000,0.000000,5.750000,1.000000,0.000000,0.00000,0.050000
50%,1.000000,1.000000,0.0,0.000000,0.000000,0.000000,11.000000,1.000000,0.000000,0.00000,0.105556
75%,2.000000,2.000000,0.0,1.000000,0.000000,0.000000,15.250000,1.000000,0.000000,0.00000,0.186364
max,5.000000,3.000000,0.0,1.000000,1.000000,1.000000,32.000000,3.000000,1.000000,1.00000,0.600000


Define Feature Columns

In [21]:
feature_cols = [
    "num_loops",
    "max_loop_depth",
    "has_recursion",
    "uses_list",
    "uses_dict",
    "uses_set",
    "lines_of_code",
    "num_functions",
    "uses_append",
    "uses_pop",
    "loop_density"
]

X = df[feature_cols]


Efficiency Label Encoding

In [22]:
# -----------------------------------------
# 1. Select the target column (what we predict)
# -----------------------------------------
# This column contains categorical labels like:
# 'efficient', 'inefficient', 'suboptimal'
y_eff = df["efficiency_label"]


# -----------------------------------------
# 2. Create a LabelEncoder object
# -----------------------------------------
# LabelEncoder converts string labels into integers
# Example:
#   efficient   -> 0
#   inefficient -> 1
#   suboptimal  -> 2
le_eff = LabelEncoder()


# -----------------------------------------
# 3. Fit the encoder and transform the labels
# -----------------------------------------
# fit(): learns the unique classes and their mapping
# transform(): converts each label to its numeric value
y_eff_encoded = le_eff.fit_transform(y_eff)


# -----------------------------------------
# 4. Inspect the learned class order
# -----------------------------------------
# This tells us which integer corresponds to which label
# VERY IMPORTANT for interpreting predictions later
le_eff.classes_


array(['efficient', 'inefficient', 'suboptimal'], dtype=object)

Train and Test 

In [23]:
# --------------------------------------------------
# Split the dataset into training and testing sets
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    
    # X = input features (static code features like loops, recursion, etc.)
    X,
    
    # y_eff_encoded = numeric labels (0, 1, 2 for efficiency classes)
    y_eff_encoded,
    
    # Use 25% of the data for testing, 75% for training
    # This is a good balance for small datasets
    test_size=0.25,
    
    # Fixes randomness so results are reproducible
    # Every run will produce the same split
    random_state=42,
    
    # Ensures that each efficiency class appears
    # in both train and test sets with similar proportions
    stratify=y_eff_encoded
)


Logistic regression

In [24]:
# --------------------------------------------------
# LOGISTIC REGRESSION WITH LOOP DENSITY
# --------------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

log_reg = LogisticRegression(max_iter=1000)

log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)

print("Classification Report (LogReg + loop_density):")
print(classification_report(
    y_test,
    y_pred,
    target_names=le_eff.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report (LogReg + loop_density):
              precision    recall  f1-score   support

   efficient       0.11      0.17      0.13         6
 inefficient       0.40      0.29      0.33         7
  suboptimal       0.25      0.20      0.22         5

    accuracy                           0.22        18
   macro avg       0.25      0.22      0.23        18
weighted avg       0.26      0.22      0.24        18

Confusion Matrix:
[[1 3 2]
 [4 2 1]
 [4 0 1]]


A linear model struggles to capture the interaction between nested loops, recursion, and data structure usage.

In [25]:
# --------------------------------------------------
# RANDOM FOREST: TRAINING + EVALUATION
# --------------------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize Random Forest model
# - n_estimators: number of trees
# - random_state: reproducibility
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# 2. Train the model
rf.fit(X_train, y_train)

# 3. Predict on test data
y_pred_rf = rf.predict(X_test)

# 4. Evaluate performance
print("Classification Report (Random Forest):")
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=le_eff.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))


Classification Report (Random Forest):
              precision    recall  f1-score   support

   efficient       0.44      0.67      0.53         6
 inefficient       0.67      0.29      0.40         7
  suboptimal       0.33      0.40      0.36         5

    accuracy                           0.44        18
   macro avg       0.48      0.45      0.43        18
weighted avg       0.50      0.44      0.43        18

Confusion Matrix:
[[4 0 2]
 [3 2 2]
 [2 1 2]]


Model Selection Rationale

In this project, both Logistic Regression and Random Forest were evaluated for efficiency classification.

1. Dataset Characteristics

Total samples: 42

Test samples: 11

Multi-class classification (3 classes)

Features are low-dimensional and structurally engineered

Given the small dataset size and limited feature space, model stability and interpretability were prioritized.

2. Experimental Comparison
Model	Accuracy	Observations
Logistic Regression	~45%	More stable, better generalization
Random Forest	~27–36%	Struggled due to small dataset size

Random Forest did not outperform Logistic Regression, likely because tree-based models require more data to learn stable non-linear splits.

3. Interpretability Requirement

This project requires Explainable AI (XAI) as a core component.

Logistic Regression provides:

Clear feature coefficients

Directional influence of features

Easier translation into human-readable feedback

Random Forest, while powerful, is less transparent and harder to explain at the decision level.

4. Final Decision

Logistic Regression was selected as the final model because it:

Performed better on this dataset

Is more stable for small sample sizes

Provides interpretable feature importance

Aligns with the project’s explainability objective

Future improvements may include:

Expanding dataset size

Adding richer static code metrics

Re-evaluating non-linear models with more data

Extract Logistic Regression Coefficients

In [27]:
import pandas as pd

feature_names = feature_cols  # notebook variable

coef_df = pd.DataFrame(
    log_reg.coef_,
    columns=feature_names,
    index=le_eff.classes_
)

coef_df


,num_loops,max_loop_depth,has_recursion,uses_list,uses_dict,uses_set,lines_of_code,num_functions,uses_append,uses_pop,loop_density
efficient,0.239678,-0.552383,0.0,0.403595,0.332881,-0.226076,0.037958,-0.619340,-0.967827,-0.579812,-0.288078
inefficient,0.090881,0.543180,0.0,-0.503780,-0.287348,0.528947,-0.095367,0.592341,0.358374,-0.702644,-0.086642
suboptimal,-0.330559,0.009203,0.0,0.100185,-0.045532,-0.302870,0.057408,0.026999,0.609452,1.282456,0.374720


Sort Feature Importance Per Class

In [28]:
# --------------------------------------------------
# SORT FEATURE IMPORTANCE FOR EACH CLASS
# --------------------------------------------------

for class_name in le_eff.classes_:
    print(f"\nTop Positive Features for '{class_name}':")
    print(coef_df.loc[class_name].sort_values(ascending=False).head())

    print(f"\nTop Negative Features for '{class_name}':")
    print(coef_df.loc[class_name].sort_values().head())



Top Positive Features for 'efficient':
uses_list        0.403595
uses_dict        0.332881
num_loops        0.239678
lines_of_code    0.037958
has_recursion    0.000000
Name: efficient, dtype: float64

Top Negative Features for 'efficient':
uses_append      -0.967827
num_functions    -0.619340
uses_pop         -0.579812
max_loop_depth   -0.552383
loop_density     -0.288078
Name: efficient, dtype: float64

Top Positive Features for 'inefficient':
num_functions     0.592341
max_loop_depth    0.543180
uses_set          0.528947
uses_append       0.358374
num_loops         0.090881
Name: inefficient, dtype: float64

Top Negative Features for 'inefficient':
uses_pop        -0.702644
uses_list       -0.503780
uses_dict       -0.287348
lines_of_code   -0.095367
loop_density    -0.086642
Name: inefficient, dtype: float64

Top Positive Features for 'suboptimal':
uses_pop         1.282456
uses_append      0.609452
loop_density     0.374720
uses_list        0.100185
lines_of_code    0.057408
Nam

Convert This Into Plain-English Feedback
Generate Explanation Function

In [ ]:
"""# --------------------------------------------------
# HUMAN-READABLE EXPLANATION FUNCTION
# --------------------------------------------------

def generate_human_explanation(model, sample_features, feature_names, label_encoder):
    
    predicted_class_index = model.predict([sample_features])[0]
    predicted_class_name = label_encoder.classes_[predicted_class_index]
    
    class_coefficients = model.coef_[predicted_class_index]
    contributions = sample_features * class_coefficients
    
    contribution_dict = dict(zip(feature_names, contributions))
    
    sorted_contributions = sorted(
        contribution_dict.items(),
        key=lambda x: abs(x[1]),
        reverse=True
    )
    
    print(f"\nFinal Classification: {predicted_class_name.upper()}\n")
    print("Reasoning:\n")
    
    for feature, value in sorted_contributions[:3]:
        
        if feature == "max_loop_depth" and sample_features[feature_names.index(feature)] >= 2:
            print("- Nested loops were detected, increasing time complexity.")
        
        elif feature == "num_loops" and sample_features[feature_names.index(feature)] >= 1:
            print("- Multiple loops contribute to higher computational cost.")
        
        elif feature == "has_recursion" and sample_features[feature_names.index(feature)] == 1:
            print("- Recursion is present, which may increase time or space complexity.")
        
        elif feature == "loop_density":
            print("- The concentration of loops relative to code length influenced the decision.")
        
        elif feature == "lines_of_code":
            print("- Overall code length slightly influenced the efficiency classification.")
        
        elif feature == "num_functions":
            print("- Multiple functions impact structural complexity.")
        
        else:
            print(f"- {feature} influenced the decision.")"""


'# --------------------------------------------------\n# HUMAN-READABLE EXPLANATION FUNCTION\n# --------------------------------------------------\n\ndef generate_human_explanation(model, sample_features, feature_names, label_encoder):\n\n    predicted_class_index = model.predict([sample_features])[0]\n    predicted_class_name = label_encoder.classes_[predicted_class_index]\n\n    class_coefficients = model.coef_[predicted_class_index]\n    contributions = sample_features * class_coefficients\n\n    contribution_dict = dict(zip(feature_names, contributions))\n\n    sorted_contributions = sorted(\n        contribution_dict.items(),\n        key=lambda x: abs(x[1]),\n        reverse=True\n    )\n\n    print(f"\nFinal Classification: {predicted_class_name.upper()}\n")\n    print("Reasoning:\n")\n\n    for feature, value in sorted_contributions[:3]:\n\n        if feature == "max_loop_depth" and sample_features[feature_names.index(feature)] >= 2:\n            print("- Nested loops were dete

The model predicts efficiency class, and then I map influential features into structured natural-language explanations to make the decision transparent.

Build a Contribution-Based Explanation

In [ ]:
"""# --------------------------------------------------
# LOCAL EXPLANATION FOR A SINGLE SAMPLE
# --------------------------------------------------

import numpy as np

def explain_prediction(model, sample_features, feature_names, label_encoder):
    
    #Generates explanation for a single prediction
    #based on logistic regression coefficients.
    
    
    # Predict class
    predicted_class_index = model.predict([sample_features])[0]
    predicted_class_name = label_encoder.classes_[predicted_class_index]
    
    # Get coefficients for predicted class
    class_coefficients = model.coef_[predicted_class_index]
    
    # Compute feature contributions
    contributions = sample_features * class_coefficients
    
    # Create readable summary
    contribution_dict = dict(zip(feature_names, contributions))
    
    # Sort by strongest positive contribution
    sorted_contributions = sorted(
        contribution_dict.items(),
        key=lambda x: abs(x[1]),
        reverse=True
    )
    
    print(f"\nPredicted Class: {predicted_class_name}\n")
    print("Top Contributing Features:")
    
    for feature, value in sorted_contributions[:3]:
        direction = "increased" if value > 0 else "decreased"
        print(f"- {feature} {direction} likelihood of this class.")"""


'# --------------------------------------------------\n# LOCAL EXPLANATION FOR A SINGLE SAMPLE\n# --------------------------------------------------\n\nimport numpy as np\n\ndef explain_prediction(model, sample_features, feature_names, label_encoder):\n\n    #Generates explanation for a single prediction\n    #based on logistic regression coefficients.\n\n\n    # Predict class\n    predicted_class_index = model.predict([sample_features])[0]\n    predicted_class_name = label_encoder.classes_[predicted_class_index]\n\n    # Get coefficients for predicted class\n    class_coefficients = model.coef_[predicted_class_index]\n\n    # Compute feature contributions\n    contributions = sample_features * class_coefficients\n\n    # Create readable summary\n    contribution_dict = dict(zip(feature_names, contributions))\n\n    # Sort by strongest positive contribution\n    sorted_contributions = sorted(\n        contribution_dict.items(),\n        key=lambda x: abs(x[1]),\n        reverse=True\n 

Test It On One Sample

In [ ]:
"""sample_series = X_test.iloc[0]
generate_human_explanation(
    log_reg,
    sample_series.values,
    FEATURE_COLS,
    le_eff
)"""


'sample_series = X_test.iloc[0]\ngenerate_human_explanation(\n    log_reg,\n    sample_series.values,\n    FEATURE_COLS,\n    le_eff\n)'

In [29]:
import joblib

joblib.dump(log_reg, "../backend/models/efficiency_model.pkl")
joblib.dump(le_eff, "../backend/models/efficiency_label_encoder.pkl")

print("Efficiency model correctly saved.")


Efficiency model correctly saved.


Pattern Classification

In [30]:
# --------------------------------------------------
# PATTERN CLASSIFICATION
# --------------------------------------------------

y_pattern = df["pattern_grouped"]

from sklearn.preprocessing import LabelEncoder

le_pattern = LabelEncoder()
y_pattern_encoded = le_pattern.fit_transform(y_pattern)

le_pattern.classes_


array(['brute_force', 'dynamic_programming', 'hashing', 'other',
       'recursion', 'stack', 'two_pointer'], dtype=object)

Train & Test

In [31]:
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X,
    y_pattern_encoded,
    test_size=0.25,
    random_state=42
)


Train Logistic Regression

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

pattern_model = LogisticRegression(max_iter=1000)

pattern_model.fit(X_train_p, y_train_p)

y_pred_p = pattern_model.predict(X_test_p)

print("Pattern Classification Report:")
print(classification_report(
    y_test_p,
    y_pred_p,
    labels=sorted(set(y_test_p)),
    target_names=le_pattern.inverse_transform(sorted(set(y_test_p)))
))


print("Confusion Matrix:")
print(confusion_matrix(y_test_p, y_pred_p))


Pattern Classification Report:
                     precision    recall  f1-score   support

        brute_force       0.50      1.00      0.67         2
dynamic_programming       0.00      0.00      0.00         3
            hashing       1.00      1.00      1.00         3
              other       0.40      0.33      0.36         6
          recursion       0.50      0.67      0.57         3
              stack       1.00      1.00      1.00         1

          micro avg       0.59      0.56      0.57        18
          macro avg       0.57      0.67      0.60        18
       weighted avg       0.49      0.56      0.51        18

Confusion Matrix:
[[2 0 0 0 0 0 0]
 [0 0 0 3 0 0 0]
 [0 0 3 0 0 0 0]
 [2 0 0 2 2 0 0]
 [0 0 0 0 2 0 1]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0]]


c:\Users\Sankarsh Nellutla\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Sankarsh Nellutla\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Sankarsh Nellutla\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to co

In [33]:
df["efficiency_label"].value_counts()


efficiency_label
inefficient    27
efficient      23
suboptimal     22
Name: count, dtype: int64

In [34]:
import joblib

joblib.dump(pattern_model, "../backend/models/pattern_model.pkl")
joblib.dump(le_pattern, "../backend/models/pattern_label_encoder.pkl")

print("Pattern model saved.")


Pattern model saved.


In [35]:
print(X.columns)


Index(['num_loops', 'max_loop_depth', 'has_recursion', 'uses_list',
       'uses_dict', 'uses_set', 'lines_of_code', 'num_functions',
       'uses_append', 'uses_pop', 'loop_density'],
      dtype='object')


In [36]:
df["pattern_grouped"].value_counts()


pattern_grouped
other                  19
brute_force            11
recursion              10
hashing                10
stack                   9
dynamic_programming     8
two_pointer             5
Name: count, dtype: int64